# Faster speech

> Naive speed-ups and six time-stretchers against Kokoro's own speed control, from 1.5x to 3x.

Notebook 100 found that Kokoro's `speed` drops sounds once it goes past about 1.5x, because 25 ms frames are too coarse
for fast speech. The alternative is to render at 1x and time-stretch the audio. audio-reader gave up on that because
it sounded smeared, and notebook 100 only tried librosa's phase vocoder. But "time-stretching" is a family of
algorithms, and they fail in very different ways. So here are three naive methods and six real ones, judged two ways: Whisper's word recall from
notebook 100 for intelligibility, and UTMOS from notebook 04 for quality.

This notebook is marked `skip_exec`: it builds a C library and takes about fifteen minutes, so `nbdev-test` skips it.

## The contenders

First, the naive recipe: render at 1x, play it faster, and correct the pitch.

- **Resample only.** Playing faster shortens the audio and raises the pitch and the formants by the same factor: the
  chipmunk. Not a candidate, just the starting point.
- **Resample, then pitch-shift down.** The literal recipe. But a pitch shift that keeps the length is itself a
  time-stretch plus a resample (librosa's uses the phase vocoder), so this is time-stretching in disguise.
- **Naive overlap-add.** Chop the audio into fixed 43 ms windows, read them further apart than they're written, and
  add them up. No alignment of any kind, so neighbouring windows meet out of phase.

Then six real time-stretchers:

- **Phase vocoder** (librosa). Take an STFT, space the frames closer together, and fix up each frequency's phase.
  Phases drift apart between frequencies, which gives speech a phasey, reverberant smear and softens consonants.
- **WSOLA** (audiotsm). Cut the waveform into overlapping windows and drop some, choosing each cut point where the
  waveform lines up with the previous window so nothing clicks.
- **HPTSM** (pytsmod). Split the sound into harmonic and percussive parts, phase-vocode the harmonics and
  overlap-add the clicks and hisses, then mix them back.
- **TD-PSOLA** (pytsmod). Track the pitch, centre a window on every glottal pulse, and drop whole pitch periods.
  The classic speech method.
- **Rubber Band**, R2 "faster" and R3 "finer" engines (pylibrb). A phase vocoder with transient detection and phase
  locking, and the industry standard. GPL, or a commercial licence.
- **Sonic**. Also drops whole pitch periods, but finds the period itself rather than needing a pitch tracker. It was
  written for blind listeners who run screen readers at several times normal speed, and Android's playback speed
  control uses a port of it. Apache-licensed, one C file, vendored in `vendor/sonic`.

In [ ]:
import ctypes
import subprocess
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import pylibrb
import pytsmod
import torch
from audiotsm import wsola
from audiotsm.io.array import ArrayReader, ArrayWriter
from IPython.display import Audio, display

from text_to_speech.data import metadata
from text_to_speech.pretrained import SR, keep_lead_in, recall, render, say

/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/nn/modules/rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Sonic has no Python package, so we compile it and call its one-shot function through `ctypes`:

In [ ]:
SONIC = Path("../vendor/sonic")
subprocess.run(["cc", "-O2", "-shared", "-fPIC", SONIC / "sonic.c", "-o", SONIC / "libsonic.dylib"], check=True)
libsonic = ctypes.CDLL(str(SONIC / "libsonic.dylib"))
c_float_p = ctypes.POINTER(ctypes.c_float)
# samples, n, speed, pitch, rate, volume, use chord pitch, sample rate, channels
libsonic.sonicChangeFloatSpeed.argtypes = [c_float_p, ctypes.c_int, *[ctypes.c_float] * 4, *[ctypes.c_int] * 3]


def sonic(x, speed):
    "Sonic works in place and returns the new length."
    buf = np.zeros(len(x) + SR, dtype=np.float32)
    buf[: len(x)] = x
    n = libsonic.sonicChangeFloatSpeed(buf.ctypes.data_as(c_float_p), len(x), speed, 1, 1, 1, 0, SR, 1)
    return buf[:n]

In [ ]:
def resample_only(x, speed):
    "Play it faster: shorter, and higher by the same factor."
    return librosa.resample(x, orig_sr=SR * speed, target_sr=SR)


def resample_then_pitch_down(x, speed):
    "Speed up by resampling, then pitch-shift back down."
    return librosa.effects.pitch_shift(resample_only(x, speed), sr=SR, n_steps=-12 * np.log2(speed))


def naive_ola(x, speed, win=1024):
    "Fixed Hann windows, read `speed` times further apart than they are written. No alignment."
    hop_out = win // 2
    hop_in = round(hop_out * speed)
    w = np.hanning(win)
    n = (len(x) - win) // hop_in + 1
    out = np.zeros(n * hop_out + win, dtype=np.float32)
    for i in range(n):
        out[i * hop_out : i * hop_out + win] += x[i * hop_in : i * hop_in + win] * w
    return out


def phase_vocoder(x, speed):
    return librosa.effects.time_stretch(x, rate=speed)


def wsola_audiotsm(x, speed):
    out = ArrayWriter(1)
    wsola(1, speed=speed).run(ArrayReader(x[None].astype(np.float32)), out)
    return out.data[0]


def hptsm(x, speed):
    return pytsmod.hptsm(x, 1 / speed)


def tdpsola(x, speed, hop=120):
    "TD-PSOLA needs a pitch track: pYIN every 5 ms, 0 where unvoiced."
    f0, _, _ = librosa.pyin(x, fmin=60, fmax=500, sr=SR, frame_length=1024, hop_length=hop)
    return pytsmod.tdpsola(x, SR, np.nan_to_num(f0), alpha=1 / speed, p_hop_size=hop, p_win_size=1024)


def rubberband(engine):
    def stretch(x, speed):
        options = pylibrb.Option.PROCESS_OFFLINE | engine | pylibrb.Option.FORMANT_PRESERVED
        stretcher = pylibrb.RubberBandStretcher(
            sample_rate=SR, channels=1, options=options, initial_time_ratio=1 / speed
        )
        x = x[None].astype(np.float32)
        stretcher.study(x, final=True)
        stretcher.process(x, final=True)
        return stretcher.retrieve_available()[0]

    return stretch


stretchers = {
    "resample only": resample_only,
    "resample, then pitch down": resample_then_pitch_down,
    "naive overlap-add": naive_ola,
    "phase vocoder": phase_vocoder,
    "WSOLA": wsola_audiotsm,
    "HPTSM": hptsm,
    "TD-PSOLA": tdpsola,
    "Rubber Band R2": rubberband(pylibrb.Option.ENGINE_FASTER),
    "Rubber Band R3": rubberband(pylibrb.Option.ENGINE_FINER),
    "Sonic": sonic,
}

The same ten sentences as notebook 100, rendered once at 1x and stretched by each method. Kokoro's own speed, keeping
the lead-in silence, is the reference.

In [ ]:
utmos = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).eval()


def mos(audio):
    with torch.no_grad():
        return utmos(torch.tensor(audio, dtype=torch.float32)[None], SR).item()


text = "Speed comes from the voice itself, not from stretching the audio afterwards."
texts = [text, *[t for t in metadata().query("id.str.startswith('LJ050')").normalized_text if len(t) < 200][:9]]
phonemes = [say(t).phonemes for t in texts]
normal = [render(ps, keep_lead_in(1)) for ps in phonemes]


def score(name, speed, audios):
    return {
        "method": name,
        "speed": speed,
        "actual": sum(map(len, normal)) / sum(map(len, audios)),
        "recall": np.mean([recall(t, a) for t, a in zip(texts, audios)]),
        "mos": np.mean([mos(a) for a in audios]),
    }


rows, examples = [score("Kokoro", 1, normal)], {}
for speed in [1.5, 2, 2.5, 3]:
    methods = {"Kokoro's speed": [render(ps, keep_lead_in(speed)) for ps in phonemes]}
    methods |= {name: [stretch(a, speed) for a in normal] for name, stretch in stretchers.items()}
    rows += [score(name, speed, audios) for name, audios in methods.items()]
    examples[speed] = {name: audios[0] for name, audios in methods.items()}
results = pd.DataFrame(rows)
results.pivot(index="method", columns="speed", values=["recall", "mos"]).round(2)

Using cache found in /Users/henrydashwood/.cache/torch/hub/tarepan_SpeechMOS_v1.2.0
/Users/henrydashwood/text-to-speech/.venv/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (i

recall                           mos                        
speed             1.0   1.5   2.0   2.5   3.0   1.0   1.5   2.0   2.5   3.0
method                                                                     
HPTSM             NaN  0.98  0.98  0.95  0.81   NaN  3.55  1.69  1.35  1.31
Kokoro           0.98   NaN   NaN   NaN   NaN  4.52   NaN   NaN   NaN   NaN
Kokoro's speed    NaN  0.94  0.88  0.74  0.73   NaN  4.37  4.23  4.02  3.98
Rubber Band R2    NaN  0.98  0.95  0.95  0.75   NaN  2.51  1.43  1.30  1.28
Rubber Band R3    NaN  0.98  0.98  0.94  0.90   NaN  3.68  2.05  1.34  1.30
Sonic             NaN  0.98  0.98  0.94  0.87   NaN  4.28  2.99  1.38  1.33
TD-PSOLA          NaN  0.98  0.98  0.93  0.82   NaN  3.73  1.61  1.35  1.31
WSOLA             NaN  0.98  0.95  0.70  0.16   NaN  4.06  2.72  1.32  1.31
phase vocoder     NaN  0.96  0.86  0.57  0.23   NaN  1.31  1.31  1.32  1.30

Speeds actually reached, where they differ from what was asked:

In [ ]:
results[(results.actual - results.speed).abs() > 0.03].round(2)

,method,speed,actual,recall,mos
9,Kokoro's speed,2.0,1.73,0.88,4.23
17,Kokoro's speed,2.5,1.94,0.74,4.02
19,WSOLA,2.5,3.05,0.70,1.32
25,Kokoro's speed,3.0,2.03,0.73,3.98
27,WSOLA,3.0,4.06,0.16,1.31


Our sentence at 2x, three ways. Then Sonic at every speed.

In [ ]:
for name in ["phase vocoder", "Kokoro's speed", "Sonic"]:
    print(name)
    display(Audio(examples[2][name], rate=SR))
for speed in [1.5, 2, 2.5, 3]:
    print(f"Sonic {speed}x")
    display(Audio(examples[speed]["Sonic"], rate=SR))

phase vocoder


Kokoro's speed


Sonic


Sonic 1.5x


Sonic 2x


Sonic 2.5x


Sonic 3x


## Results

On 1 November 2026, word recall (Whisper) and predicted MOS (UTMOS) over the ten sentences. At 1x Kokoro scores
97.6% and 4.52.

| method | 1.5x recall | 1.5x MOS | 2x recall | 2x MOS | 2.5x recall | 3x recall |
|---|---|---|---|---|---|---|
| Kokoro's speed (reaches 1.51x, 1.73x, 1.94x, 2.03x) | 94.1% | **4.36** | 87.3% | **4.22** | 73.9% | 73.3% |
| resample only | 97.6% | 1.98 | 72.7% | 1.27 | 10.7% | 0.8% |
| resample, then pitch down | 90.7% | 1.30 | 33.8% | 1.30 | 7.6% | 7.2% |
| naive overlap-add | 97.6% | 3.47 | 97.1% | 2.06 | **94.0%** | 78.4% |
| phase vocoder | 95.1% | 1.32 | 86.0% | 1.31 | 52.8% | 15.8% |
| WSOLA (overshoots: 3.05x, 4.06x) | 97.6% | 4.03 | 92.8% | 2.62 | 62.6% | 14.6% |
| HPTSM | 97.6% | 3.52 | 97.6% | 1.71 | **94.6%** | 77.7% |
| TD-PSOLA | 97.6% | 3.76 | 97.6% | 1.56 | 93.7% | 82.5% |
| Rubber Band R2 | 97.6% | 2.51 | 95.1% | 1.45 | 92.1% | 74.4% |
| Rubber Band R3 | 97.6% | 3.69 | 97.6% | 2.05 | 93.1% | **89.9%** |
| **Sonic** | **97.6%** | 4.29 | **97.6%** | 2.92 | 93.7% | 88.8% |

- **The naive recipe is the worst of all.** Resampling then pitch-shifting back loses two thirds of the words at
  2x, worse than not correcting the pitch at all. librosa's pitch shift is a phase-vocoder stretch plus a resample, so
  it smears, and the resampling moves the formants too.
- **Naive overlap-add is surprisingly good**: as intelligible as Sonic up to 2.5x. But it sounds rougher (2.06 against
  2.92 at 2x), because its windows meet out of phase, and by 3x it loses a fifth of the words to Sonic's tenth. Lining
  the cuts up on pitch periods is exactly what Sonic adds, and that's what the gap measures.
- **The phase vocoder is the worst real stretcher**, with a MOS of 1.3 already at 1.5x. That's the smear audio-reader
  heard, and it's what notebook 100 compared against. Time-stretching wasn't the problem; this time-stretcher was.
- **Sonic is the best overall.** At 2x, Whisper hears every word it heard at 1x, and UTMOS prefers it to every other
  stretcher. At 3x it is about a point behind the best. It also costs almost nothing: all ten sentences, at any
  speed, in under a twentieth of a second.
- **Kokoro's own speed sounds the most natural to UTMOS**, but it never gets past 2x, and it loses words the
  stretchers keep.
- **UTMOS hits its floor**, about 1.3, for every stretcher from 2.5x. It judges fast speech as bad speech, whatever
  the method, so above 2x only recall and your ears can tell methods apart.
- audiotsm's WSOLA overshoots the speed it is asked for from 2.5x, so its numbers there aren't comparable.

So the player uses Sonic for speed. It suits the phones too: Android's own speed control already uses a port of it,
and on iOS it compiles as-is. Rubber Band, the only other contender at 3x, would need a commercial licence for the
App Store.

Whisper and UTMOS are not ears. Listen to the examples above before believing the table, especially at 3x.

## Things to try

- Kokoro's own speed up to 1.3x, where it rounds few phonemes down, then Sonic for the rest. Does UTMOS prefer it?
- Sonic has a higher-quality mode (`sonicSetQuality` on its streaming interface). Is it audibly better at 3x?
- audio-reader's listeners chose speeds by ear. Find the speed where each method stops being comfortable to listen to
  for a whole paragraph.